# Declarative routing at 27B: Qwen3.8 27B, `read` against `generate`

Question: at 27B, does a stated region declaration (`generate`) catch up with the logit read
(`read`)? The routing table shows `read` ahead of `generate` at 0.5B, 1.5B and 7B. The paper we
position against reports that stating works at 27B and 31B, so this run completes the scale
curve on the same 384 probes and layouts. See [qwen38_27b_plan.md](../docs/qwen38_27b_plan.md).

## What is different about this model

- Hybrid: 64 layers, 48 Gated DeltaNet (no KV cache) and 16 full-attention layers. The
  attention-probing modes assume every layer has attention and are **not run** here.
- Thinking is on by default. `chat_template_kwargs.enable_thinking: false` is set in
  `configs/kaggle_declare_27b.yaml`, otherwise the next token after the prompt starts a
  `<think>` block and the Yes/No read is measured on the wrong position.
- The checkpoint is a vision-language model. The preflight cell decides between the causal and
  image-text loaders from the config class.
- nf4 weights are about 14 to 16 GB, so one T4 is not enough. The model is split over two T4s.

## Stages

1. Preflight: GPU count, architecture, loader choice, template with thinking off, finite
   logits on a short prompt and a Yes/No sanity check. Costs minutes and no probe data.
2. Timing: 8 probes, both modes. Prints the projected hours for 384 probes. Stops if the
   projection exceeds `MAX_HOURS` so a slow DeltaNet fallback is found early.
3. Grid: one seed per session, pinned to `results/declare/gold/gold384_s<seed>.json`.
4. Report: merges the committed 0.5B, 1.5B and 7B summaries so the table shows all four scales.

## Stop rules

Preflight failing, non-finite logits, or the Yes/No sanity check failing means the model is
being measured wrongly: stop and fix, do not run probes.


In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'
FRESH = True
REQUIRE = [('sleep/lm.py', 'chat_template_kwargs'), ('declare/run.py', 'gold-ref'), ('eval/declare_report.py', 'paired')]

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

if FRESH and GIT_URL and os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO], check=True)
    print(f'removed {REPO} so the clone is current')

if not looks_like_source(REPO) and GIT_URL:
    r = subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)

if not looks_like_source(REPO):
    print('no clone; falling back to attached inputs')
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)

assert looks_like_source(REPO), (
    'No source found. Either (a) set GIT_URL above, or (b) run scripts/package_source.py '
    'locally, upload the zip as a Kaggle Dataset, and attach it via Add Input. '
    f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)

_h = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True, cwd=REPO)
print('source commit:', _h.stdout.strip() or 'unknown (not a git checkout)')

_stale = [f for f, token in REQUIRE
          if not os.path.exists(f) or token not in open(f, encoding='utf-8').read()]
assert not _stale, (
    f'This checkout is older than the seed support these cells need: {_stale}. '
    'It came from an attached dataset or a leftover /kaggle/working/myrios rather than a '
    'fresh clone. Detach any source dataset under Add Input, keep only the runs_declare.zip '
    'dataset, and re-run this cell with FRESH = True.')
print('source is current for the 27B run')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass

exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Re-run only the next cell if the kernel restarts; it re-derives everything.

- `SEEDS`: one seed per session
- `MAX_HOURS`: the timing cell stops above this projection


In [ ]:
import os, sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.46.1'], check=True)
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

CFG = 'configs/kaggle_declare_27b.yaml'
RUNS = '/kaggle/working/artifacts/runs_declare_27b'
MERGED = '/kaggle/working/artifacts/runs_declare_all'
DATA = 'artifacts/data/hotpotqa_spread'
N_EVAL = 96
PER_TRAJECTORY = 4
SEEDS = [0]
MODES = 'read generate'
MAX_HOURS = 3.5
MODEL = 'Qwen/Qwen3.8-27B'

from common.io import load_config
_c = load_config(CFG)
assert _c['data']['dir'] == DATA, f"data.dir mismatch: {_c['data']['dir']} != {DATA}"
assert _c['model']['base'] == MODEL
assert _c['model']['chat_template_kwargs'] == {'enable_thinking': False}
os.makedirs(RUNS, exist_ok=True)
print(f'seeds {SEEDS}, modes {MODES}, model {MODEL}')


## Preflight

Reads only the config and a short forward pass. The loader is chosen from whether the config
class is registered for causal LM. Nothing here touches probe data.


In [ ]:
import torch
from collections import Counter
from transformers import AutoConfig, AutoTokenizer
from transformers.models.auto.modeling_auto import MODEL_FOR_CAUSAL_LM_MAPPING

n_gpu = torch.cuda.device_count()
print('gpus', n_gpu)
assert n_gpu >= 2, 'need two T4s: set Session options -> Accelerator -> GPU T4 x2'

cfg_hf = AutoConfig.from_pretrained(MODEL)
text = getattr(cfg_hf, 'text_config', cfg_hf)
print('config class', type(cfg_hf).__name__, 'architectures', getattr(cfg_hf, 'architectures', None))
kinds = Counter(getattr(text, 'layer_types', []) or [])
print('layers', getattr(text, 'num_hidden_layers', None), 'layer types', dict(kinds))
LOADER = 'causal' if type(cfg_hf) in MODEL_FOR_CAUSAL_LM_MAPPING else 'image_text'
print('loader chosen:', LOADER)

tok = AutoTokenizer.from_pretrained(MODEL)
sample = [{'role': 'user', 'content': 'hi'}]
on = tok.apply_chat_template(sample, tokenize=False, add_generation_prompt=True)
off = tok.apply_chat_template(sample, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print('thinking default tail:', repr(on[-40:]))
print('thinking off tail    :', repr(off[-40:]))
assert on != off, 'enable_thinking changes nothing in this template, so it is not being honoured'


Loads the weights once to run the sanity checks. This is the slow step (download plus 4-bit
quantization), so the same session then continues straight into timing and the grid.


In [ ]:
SANITY_SRC = '''import sys, torch
sys.path.insert(0, '/kaggle/working/myrios')
from common.io import load_config
from sleep.lm import load_backbone, chat_text
from declare.elicit import register_repeat, check_probe_parity

cfg = load_config(sys.argv[1], {'model': {'loader': sys.argv[2]}})
impl = register_repeat()
model, tokenizer = load_backbone(cfg, attn_implementation=impl)
print('devices', sorted({str(p.device) for p in model.parameters()}))

prompt = chat_text(tokenizer, None, 'Answer with only Yes or No. Is Paris the capital of France?')
enc = tokenizer([prompt], return_tensors='pt').to(next(model.parameters()).device)
with torch.no_grad():
    logits = model(**enc).logits[0, -1].float()
assert torch.isfinite(logits).all(), 'non-finite logits: fp16 overflow, this run would be garbage'
top = tokenizer.decode(logits.argmax().item())
print('next token after the prompt:', repr(top))
assert top.strip().lower() in ('yes', 'no'), 'first token is not Yes/No, thinking may still be on'

diff, same_top, ok = check_probe_parity(model, tokenizer)
print(f'repeat attention vs stock sdpa: max logprob diff {diff:.4f}, same top-1 {same_top}')
assert ok, 'the repeat attention path does not reproduce stock sdpa on this model'
'''
open('/kaggle/working/sanity_c3.py', 'w').write(SANITY_SRC)
run(f'python /kaggle/working/sanity_c3.py {CFG} {LOADER}')


In [ ]:
run(f"python data/load_hotpotqa.py --n-train 8 --n-eval {N_EVAL} --per-trajectory {PER_TRAJECTORY} --early-frac 1.0 --out {DATA}")


## Timing

Runs 4 and then 12 probes in both modes and takes the difference, so the model load time
cancels. The probes are a prefix of the real layout but use no gold reference.


In [ ]:
import time
def timed(n):
    t0 = time.time()
    run(f"python declare/run.py --config {CFG} --modes {MODES} --limit {n} "
        f"--out {RUNS}/timing{n} --set model.loader={LOADER}")
    return time.time() - t0

small, large = timed(4), timed(12)
per_probe = (large - small) / 8
hours = per_probe * 384 / 3600
print(f'{small:.0f}s for 4 probes, {large:.0f}s for 12: {per_probe:.1f} s per probe, projected {hours:.1f} h for 384')
if hours > MAX_HOURS:
    raise RuntimeError(
        f'projected {hours:.1f} h exceeds MAX_HOURS {MAX_HOURS}: the DeltaNet fallback kernel is '
        f'likely slow. Install flash-linear-attention, or cut modes, before spending the session.')


## The grid

One seed per session. Each run is pinned to the committed gold distribution for the seed.


In [ ]:
for seed in SEEDS:
    gold = f'{REPO}/results/declare/gold/gold384_s{seed}.json'
    out = f'{RUNS}/Qwen3.8-27B_384_s{seed}'
    if os.path.exists(f'{out}/read.summary.json') and os.path.exists(f'{out}/generate.summary.json'):
        print('skip', out)
        continue
    print('=' * 70)
    print(f'Qwen3.8-27B seed {seed}', flush=True)
    run(f"python declare/run.py --config {CFG} --modes {MODES} --out {out} --gold-ref {gold} "
        f"--set model.loader={LOADER} seed={seed}")
    run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_declare_27b.zip")


## Report


In [ ]:
import shutil
from pathlib import Path
if os.path.exists(MERGED):
    shutil.rmtree(MERGED)
shutil.copytree(f'{REPO}/results/declare', MERGED, ignore=shutil.ignore_patterns('gold'))
for d in sorted(Path(RUNS).glob('Qwen3.8-27B_384_s*')):
    shutil.copytree(d, f'{MERGED}/{d.name}')
run(f"python eval/declare_report.py --run-root {MERGED} --out docs/results_declare_27b.md")
from IPython.display import Markdown, display
display(Markdown(open('docs/results_declare_27b.md').read()))
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_declare_27b.zip")
print()
print('=' * 64)
print('DOWNLOAD /kaggle/working/runs_declare_27b.zip (right panel -> Output)')
print('=' * 64)
